In [1]:
import importlib, subprocess, sys

REQUIRED = [
    ("lightgbm",      "lightgbm>=3.3"),
    ("mlflow",         "mlflow"),
    ("pyarrow",        "pyarrow"),
    ("sklearn",        "scikit-learn"),
]

for mod, pkg in REQUIRED:
    if importlib.util.find_spec(mod) is None:
        print(f"Instalando {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"{pkg} instalado.")
    else:
        import importlib.metadata
        try:
            ver = importlib.metadata.version(pkg.split(">=")[0].split("==")[0])
        except Exception:
            ver = "ok"
        print(f"{mod}: {ver}")

print("Setup OK")

lightgbm: 4.7.0
mlflow: ok
pyarrow: 24.0.0
sklearn: 1.8.0
Setup OK


In [2]:
import socket

def port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1)
        return s.connect_ex(('127.0.0.1', port)) == 0

import subprocess, time
import mlflow

if not port_open(6000):
    proc = subprocess.Popen(['bash', '/srv/start_mlflow.sh'])
    print(f'MLflow PID: {proc.pid}')
    print('MLflow iniciado — esperando 4s...')
    time.sleep(4)

mlflow.set_tracking_uri('http://localhost:6000')
mlflow.set_experiment('latam2cloud')
print('MLflow OK — http://localhost:6000  |  experimento: latam2cloud')

MLflow OK — http://localhost:6000  |  experimento: latam2cloud


In [3]:

import os, gc, pickle, warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import lightgbm as lgb
from sklearn.metrics import cohen_kappa_score, f1_score, balanced_accuracy_score, confusion_matrix
warnings.filterwarnings('ignore')

SAVE_DIR   = '/srv/work/models'
IN_FEAT_V5 = '/srv/work/te_features_v5.parquet'
IN_FEAT_V6 = '/srv/work/te_features_v6.parquet'
ROUTE_KEYS = ['source_agent', 'cloud_provider', 'region']
ID_COLS    = ROUTE_KEYS + ['window']
SPLIT_COL  = 'split'
TARGET_COL = 'label_ventana'
HORIZON    = 2

W_RF_V9  = 0.80
W_LGB_V9 = 0.20

BBSE_REG = 1e-3

WS_N_TREES = 100
WS_LR      = 0.005
N_SUBSAMPLE_WS = 1_500_000

QWK_FLOOR    = 0.65
RECALTA_FLOOR = 0.55

LABEL_COLS = {0: 'Normal', 1: 'Baja', 2: 'Media', 3: 'Alta'}

print(f'LightGBM {lgb.__version__}')
print(f'BBSE_REG={BBSE_REG}  |  WS_N_TREES={WS_N_TREES}  |  WS_LR={WS_LR}')
print(f'Floors: QWK>={QWK_FLOOR} AND RecAlta>={RECALTA_FLOOR}')

LightGBM 4.7.0
BBSE_REG=0.001  |  WS_N_TREES=100  |  WS_LR=0.005
Floors: QWK>=0.65 AND RecAlta>=0.55


In [4]:

def evaluar(nombre, y_true, proba, verbose=True):
    y_pred = np.argmax(proba, axis=1)
    qwk    = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    cm     = confusion_matrix(y_true, y_pred, labels=[0,1,2,3])
    rec_a  = cm[3,3] / cm[3].sum() if cm[3].sum() > 0 else 0.0
    mf1    = f1_score(y_true, y_pred, average='macro', zero_division=0)
    bacc   = balanced_accuracy_score(y_true, y_pred)
    cand   = qwk >= QWK_FLOOR and rec_a >= RECALTA_FLOOR
    if verbose:
        mark_q = '✅' if qwk >= QWK_FLOOR else '✗ '
        mark_r = '✅' if rec_a >= RECALTA_FLOOR else '✗ '
        print(f'  {"-"*80}')
        print(f'  {nombre}')
        print(f'  {"-"*80}')
        print(f'  QWK      = {qwk:.4f}  {mark_q}  (floor>={QWK_FLOOR})')
        print(f'  RecAlta  = {rec_a:.4f}  {mark_r}  (floor>={RECALTA_FLOOR})')
        print(f'  MacroF1  = {mf1:.4f}')
        print(f'  BalAcc   = {bacc:.4f}')
        print(f'  CANDIDATO: {"SI" if cand else "NO"}')
        for i, row in enumerate(cm):
            print(f'    {LABEL_COLS[i]:<8}: {str(row.tolist()):>60}')
    return {'qwk': qwk, 'rec_alta': rec_a, 'macro_f1': mf1, 'bacc': bacc,
            'candidato': cand, 'cm': cm}


def evaluar_t1(nombre, y_true, proba, T):
    """T1 threshold: predict Alta si P(Alta|x)>=T, else argmax(otras)."""
    y_pred = np.where(
        proba[:, 3] >= T,
        3,
        np.argmax(proba[:, :3], axis=1)
    )
    qwk   = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    cm    = confusion_matrix(y_true, y_pred, labels=[0,1,2,3])
    rec_a = cm[3,3] / cm[3].sum() if cm[3].sum() > 0 else 0.0
    mf1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    cand  = qwk >= QWK_FLOOR and rec_a >= RECALTA_FLOOR
    return {'qwk': qwk, 'rec_alta': rec_a, 'macro_f1': mf1, 'candidato': cand}


print('evaluar() y evaluar_t1() definidas.')

evaluar() y evaluar_t1() definidas.


In [5]:

print('Cargando ens_v9_val_proba.npy + lgb_v7_val_proba.npy (v5 val)...')
ens_v9_proba_v5 = np.load(f'{SAVE_DIR}/ens_v9_val_proba.npy').astype('float32')
lgb_v7_proba_v5 = np.load(f'{SAVE_DIR}/lgb_v7_val_proba.npy').astype('float32')

print('Cargando y_val_v5 desde te_features_v5.parquet...')
keep_v5 = ROUTE_KEYS + [TARGET_COL, SPLIT_COL]
df_v5   = pq.read_table(IN_FEAT_V5, columns=keep_v5).to_pandas()
df_v5['y'] = df_v5.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
y_val_v5 = df_v5.loc[df_v5[SPLIT_COL] == 'val', 'y'].dropna().values.astype(int)
del df_v5; gc.collect()

n_v5 = min(len(y_val_v5), ens_v9_proba_v5.shape[0], lgb_v7_proba_v5.shape[0])
y_val_v5        = y_val_v5[:n_v5]
ens_v9_proba_v5 = ens_v9_proba_v5[:n_v5]
lgb_v7_proba_v5 = lgb_v7_proba_v5[:n_v5]

print(f'  ens_v9_proba_v5 : {ens_v9_proba_v5.shape}')
print(f'  lgb_v7_proba_v5 : {lgb_v7_proba_v5.shape}')
print(f'  y_val_v5        : {y_val_v5.shape}')
for k, name in LABEL_COLS.items():
    n = (y_val_v5 == k).sum()
    print(f'  {name:<8}: {n:>9,} ({100*n/len(y_val_v5):.1f}%)')

res_v9_argmax = evaluar('Baseline: v9 Ens argmax (v5 val)', y_val_v5, ens_v9_proba_v5)
res_v9_t138   = evaluar_t1('Candidato v14 R5: v9+T1=0.38 (v5 val)', y_val_v5, ens_v9_proba_v5, T=0.38)
print(f'\n  v14 R5 T=0.38: QWK={res_v9_t138["qwk"]:.4f}  RecAlta={res_v9_t138["rec_alta"]:.4f}  CAND={res_v9_t138["candidato"]}')
print('Celda 3 OK')

Cargando ens_v9_val_proba.npy + lgb_v7_val_proba.npy (v5 val)...
Cargando y_val_v5 desde te_features_v5.parquet...
  ens_v9_proba_v5 : (644690, 4)
  lgb_v7_proba_v5 : (644690, 4)
  y_val_v5        : (644690,)
  Normal  :   551,191 (85.5%)
  Baja    :    58,645 (9.1%)
  Media   :    14,769 (2.3%)
  Alta    :    20,085 (3.1%)
  --------------------------------------------------------------------------------
  Baseline: v9 Ens argmax (v5 val)
  --------------------------------------------------------------------------------
  QWK      = 0.6529  ✅  (floor>=0.65)
  RecAlta  = 0.6509  ✅  (floor>=0.55)
  MacroF1  = 0.6631
  BalAcc   = 0.6665
  CANDIDATO: SI
    Normal  :                                  [524815, 15407, 4230, 6739]
    Baja    :                                   [17116, 38399, 1816, 1314]
    Media   :                                     [4888, 1895, 6029, 1957]
    Alta    :                                     [5029, 593, 1390, 13073]

  v14 R5 T=0.38: QWK=0.6497  RecAlta=0.6

In [6]:

print('Cargando te_features_v6.parquet (split=val)...')

keep_meta = ROUTE_KEYS + [TARGET_COL, SPLIT_COL]
df_v6_meta = pq.read_table(IN_FEAT_V6, columns=keep_meta,
                            filters=[('split', '=', 'val')]).to_pandas()
df_v6_meta['y'] = df_v6_meta.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
y_val_v6 = df_v6_meta.loc[df_v6_meta[SPLIT_COL] == 'val', 'y'].dropna().values.astype(int)
del df_v6_meta; gc.collect()

df_val6 = pq.read_table(IN_FEAT_V6, filters=[('split', '=', 'val')]).to_pandas()
df_val6['y'] = df_val6.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
df_val6 = df_val6.dropna(subset=['y'])
y_val_v6 = df_val6['y'].values.astype(int)

FEAT_COLS_V6 = [c for c in df_val6.columns
                if c not in ID_COLS + [TARGET_COL, SPLIT_COL, 'y']
                and not c.startswith('n_samp')]

X_val_v6 = df_val6[FEAT_COLS_V6].values.astype('float32')
del df_val6; gc.collect()

ratio_idx = [i for i, c in enumerate(FEAT_COLS_V6) if 'ratio' in c]
other_idx  = [i for i, c in enumerate(FEAT_COLS_V6) if 'ratio' not in c]
for idx in ratio_idx:
    mask = np.isnan(X_val_v6[:, idx])
    if mask.any(): X_val_v6[mask, idx] = 1.0
for idx in other_idx:
    mask = np.isnan(X_val_v6[:, idx])
    if mask.any(): X_val_v6[mask, idx] = 0.0

print(f'  X_val_v6: {X_val_v6.shape}  NaN restantes: {np.isnan(X_val_v6).sum()}')
print(f'  y_val_v6: {y_val_v6.shape}')
for k, name in LABEL_COLS.items():
    n = (y_val_v6 == k).sum()
    print(f'  {name:<8}: {n:>9,} ({100*n/len(y_val_v6):.1f}%)')
print('Celda 4 OK')

Cargando te_features_v6.parquet (split=val)...
  X_val_v6: (1138777, 100)  NaN restantes: 0
  y_val_v6: (1138777,)
  Normal  : 1,024,150 (89.9%)
  Baja    :    70,518 (6.2%)
  Media   :    22,606 (2.0%)
  Alta    :    21,503 (1.9%)
Celda 4 OK


In [7]:

import time

print('Cargando LGB v7 model...')
lgb_v7 = lgb.Booster(model_file=f'{SAVE_DIR}/lgb_v7_model.txt')
n_feat_lgb = lgb_v7.num_feature()
print(f'  lgb_v7 features: {n_feat_lgb}  |  X_val_v6 features: {X_val_v6.shape[1]}')

print('Prediciendo LGB v7 en v6 val...')
t0 = time.time()
lgb_v7_proba_v6 = lgb_v7.predict(X_val_v6[:, :n_feat_lgb]).astype('float32')
print(f'  LGB v7 v6 val inference: {time.time()-t0:.1f}s')
np.save(f'{SAVE_DIR}/lgb_v7_proba_v6_val.npy', lgb_v7_proba_v6)
print(f'  Guardado: lgb_v7_proba_v6_val.npy')

print()
res_lgb_v7_v6 = evaluar('LGB v7 argmax sobre v6 val (sin calibracion)', y_val_v6, lgb_v7_proba_v6)
res_lgb_v7_v6_t138 = evaluar_t1('LGB v7 T=0.38 sobre v6 val', y_val_v6, lgb_v7_proba_v6, T=0.38)
print(f'  LGB v7+T1=0.38 en v6: QWK={res_lgb_v7_v6_t138["qwk"]:.4f}  RecAlta={res_lgb_v7_v6_t138["rec_alta"]:.4f}')
print('Celda 5 OK')

Cargando LGB v7 model...
  lgb_v7 features: 100  |  X_val_v6 features: 100
Prediciendo LGB v7 en v6 val...
  LGB v7 v6 val inference: 112.0s
  Guardado: lgb_v7_proba_v6_val.npy

  --------------------------------------------------------------------------------
  LGB v7 argmax sobre v6 val (sin calibracion)
  --------------------------------------------------------------------------------
  QWK      = 0.5946  ✗   (floor>=0.65)
  RecAlta  = 0.4583  ✗   (floor>=0.55)
  MacroF1  = 0.6331
  BalAcc   = 0.6295
  CANDIDATO: NO
    Normal  :                                  [991461, 19170, 5181, 8338]
    Baja    :                                   [15263, 51918, 2238, 1099]
    Media   :                                     [7954, 3743, 8037, 2872]
    Alta    :                                      [8864, 939, 1846, 9854]
  LGB v7+T1=0.38 en v6: QWK=0.5864  RecAlta=0.5012
Celda 5 OK


In [8]:

print('R2: BBSE Label Shift Correction (modelo base: LGB v7)')
print('='*60)

y_pred_lgb_v5 = np.argmax(lgb_v7_proba_v5, axis=1)
cm_lgb_v5 = confusion_matrix(y_val_v5, y_pred_lgb_v5, labels=[0,1,2,3]).astype('float64')
C_source = cm_lgb_v5 / cm_lgb_v5.sum(axis=0, keepdims=True)
print('C_source (P(yhat=k|y=j) LGB v7 en v5 val):')
for j in range(4):
    print(f'  y={LABEL_COLS[j]:<8}: ' + '  '.join(f'{C_source[k,j]:.3f}' for k in range(4)))

y_pred_lgb_v6 = np.argmax(lgb_v7_proba_v6, axis=1)
mu_target = np.bincount(y_pred_lgb_v6, minlength=4).astype('float64') / len(y_pred_lgb_v6)
print(f'\nmu_target (P(yhat=k) LGB v7 en v6 val):')
for k in range(4):
    print(f'  yhat={LABEL_COLS[k]:<8}: {mu_target[k]:.4f}')

C_reg = C_source + BBSE_REG * np.eye(4)
w_bbse = np.linalg.solve(C_reg.T, mu_target)
w_bbse = np.maximum(w_bbse, 0.0)
print(f'\nBBSE weights w_k:')
for k in range(4):
    print(f'  {LABEL_COLS[k]:<8}: w={w_bbse[k]:.4f}')

proba_bbse = lgb_v7_proba_v6 * w_bbse[np.newaxis, :]
proba_bbse = (proba_bbse / proba_bbse.sum(axis=1, keepdims=True)).astype('float32')

print()
res_bbse = evaluar('R2 BBSE: LGB v7 posteriors corregidos (v6 val)', y_val_v6, proba_bbse)

print('\nR2 BBSE + T1 scan:')
best_bbse_t = None; best_bbse_rec = -1
print(f'  {"T":>5}  {"QWK":>7}  {"RecAlta":>9}  Cand')
for T in np.round(np.arange(0.05, 0.51, 0.01), 2):
    r = evaluar_t1('', y_val_v6, proba_bbse, T=T)
    cand_str = 'OK' if r['candidato'] else ''
    if r['candidato'] or (abs(T*100 % 5) < 0.5):
        print(f'  {T:>5.2f}  {r["qwk"]:>7.4f}  {r["rec_alta"]:>9.4f}  {cand_str}')
    if r['candidato'] and r['rec_alta'] > best_bbse_rec:
        best_bbse_rec = r['rec_alta']; best_bbse_t = T; best_bbse_r = r

if best_bbse_t is not None:
    print(f'\n  BBSE mejor candidato T={best_bbse_t}: QWK={best_bbse_r["qwk"]:.4f}  RecAlta={best_bbse_r["rec_alta"]:.4f}')
else:
    print('\n  BBSE: sin candidato con T1 scan')

print('\n' + '-'*40)
print('R2b: Prior reweighting (w_k = P_v6(y=k) / P_v5(y=k)):')
p_v5 = np.bincount(y_val_v5, minlength=4).astype('float64') / len(y_val_v5)
p_v6 = np.bincount(y_val_v6, minlength=4).astype('float64') / len(y_val_v6)
w_prior = p_v6 / (p_v5 + 1e-8)
print(f'  P_v5: ' + '  '.join(f'{LABEL_COLS[k]}={p_v5[k]:.4f}' for k in range(4)))
print(f'  P_v6: ' + '  '.join(f'{LABEL_COLS[k]}={p_v6[k]:.4f}' for k in range(4)))
print(f'  w:    ' + '  '.join(f'{LABEL_COLS[k]}={w_prior[k]:.3f}' for k in range(4)))

proba_prior = lgb_v7_proba_v6 * w_prior[np.newaxis, :]
proba_prior = (proba_prior / proba_prior.sum(axis=1, keepdims=True)).astype('float32')
res_prior = evaluar('R2b Prior reweighting: LGB v7 + w_prior (v6 val)', y_val_v6, proba_prior)

print('\nCelda 6 OK')

R2: BBSE Label Shift Correction (modelo base: LGB v7)
C_source (P(yhat=k|y=j) LGB v7 en v5 val):
  y=Normal  : 0.956  0.027  0.007  0.009
  y=Baja    : 0.319  0.639  0.031  0.011
  y=Media   : 0.391  0.129  0.384  0.097
  y=Alta    : 0.334  0.050  0.074  0.542

mu_target (P(yhat=k) LGB v7 en v6 val):
  yhat=Normal  : 0.8988
  yhat=Baja    : 0.0665
  yhat=Media   : 0.0152
  yhat=Alta    : 0.0195

BBSE weights w_k:
  Normal  : w=0.9582
  Baja    : w=0.0000
  Media   : w=0.0000
  Alta    : w=0.0000

  --------------------------------------------------------------------------------
  R2 BBSE: LGB v7 posteriors corregidos (v6 val)
  --------------------------------------------------------------------------------
  QWK      = 0.0000  ✗   (floor>=0.65)
  RecAlta  = 0.0000  ✗   (floor>=0.55)
  MacroF1  = 0.2368
  BalAcc   = 0.2500
  CANDIDATO: NO
    Normal  :                                           [1024150, 0, 0, 0]
    Baja    :                                             [70518, 0, 0, 0]

In [9]:

import time

print('R3: LGB v7 Warm-Start sobre v6 train (init_model = lgb_v7_model.txt)')
print('='*70)
print(f'  WS_N_TREES={WS_N_TREES}  |  WS_LR={WS_LR}  |  N_SUBSAMPLE={N_SUBSAMPLE_WS:,}')
print('  Nota: evaluación LGB_warmstart SOLO (rf_v51_model.pkl no disponible, LOG-50)')

print('\nCargando v6 train...')
df_tr = pq.read_table(IN_FEAT_V6, filters=[('split', '=', 'train')]).to_pandas()
df_tr['y'] = df_tr.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
df_tr = df_tr.dropna(subset=['y'])

X_full_ws = df_tr[FEAT_COLS_V6].values.astype('float32')
y_full_ws  = df_tr['y'].values.astype(int)
del df_tr; gc.collect()

rng_ws        = np.random.default_rng(42)
idx_alta_ws   = np.where(y_full_ws == 3)[0]
idx_noalta_ws = np.where(y_full_ws != 3)[0]
n_sub         = min(N_SUBSAMPLE_WS, len(idx_noalta_ws))
idx_sub       = rng_ws.choice(idx_noalta_ws, n_sub, replace=False)
idx_ws        = np.concatenate([idx_alta_ws, idx_sub])
rng_ws.shuffle(idx_ws)

X_train_ws = X_full_ws[idx_ws]
y_train_ws = y_full_ws[idx_ws]
del X_full_ws, y_full_ws, idx_sub, idx_ws; gc.collect()

for i, c in enumerate(FEAT_COLS_V6):
    mask = np.isnan(X_train_ws[:, i])
    if mask.any(): X_train_ws[mask, i] = 1.0 if 'ratio' in c else 0.0

print(f'  X_train_ws: {X_train_ws.shape}  NaN: {np.isnan(X_train_ws).sum()}')
for k, name in LABEL_COLS.items():
    n = (y_train_ws == k).sum()
    print(f'  {name:<8}: {n:>9,} ({100*n/len(y_train_ws):.1f}%)')

class_freq    = np.bincount(y_train_ws, minlength=4) / len(y_train_ws)
sample_weight = (1.0 / (class_freq + 1e-8))[y_train_ws]
sample_weight /= sample_weight.mean()
print(f'  Peso Alta/Normal: {sample_weight[y_train_ws==3].mean()/sample_weight[y_train_ws==0].mean():.1f}x')

n_feat_lgb = lgb_v7.num_feature()
dtrain_ws = lgb.Dataset(
    X_train_ws[:, :n_feat_lgb], label=y_train_ws,
    weight=sample_weight, free_raw_data=True
)
dval_ws = lgb.Dataset(
    X_val_v6[:, :n_feat_lgb], label=y_val_v6,
    reference=dtrain_ws, free_raw_data=True
)

ws_params = dict(
    objective='multiclass', num_class=4,
    metric='multi_logloss',
    num_leaves=63, learning_rate=WS_LR,
    feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=5,
    min_child_samples=20, n_jobs=8, random_state=42, verbose=-1,
)

print(f'\nEntrenando LGB warm-start ({WS_N_TREES} árboles adicionales, LR={WS_LR})...')
t0 = time.time()
lgb_warmstart = lgb.train(
    ws_params, dtrain_ws,
    num_boost_round = WS_N_TREES,
    valid_sets      = [dval_ws],
    init_model      = f'{SAVE_DIR}/lgb_v7_model.txt',
    callbacks       = [lgb.log_evaluation(20), lgb.early_stopping(30)],
)
elapsed = time.time() - t0
print(f'  Entrenado en {elapsed/60:.1f} min  |  best_iter extra: {lgb_warmstart.best_iteration}')
lgb_warmstart.save_model(f'{SAVE_DIR}/lgb_v7_warmstart_model.txt')
del X_train_ws, y_train_ws, dtrain_ws, dval_ws; gc.collect()

proba_lgb_ws = lgb_warmstart.predict(X_val_v6[:, :n_feat_lgb]).astype('float32')
np.save(f'{SAVE_DIR}/lgb_v7ws_v6_val_proba.npy', proba_lgb_ws)
print(f'  Guardado: lgb_v7ws_v6_val_proba.npy')

print()
res_lgb_ws = evaluar('R3 LGB warm-start (v6 val) — argmax', y_val_v6, proba_lgb_ws)

print('\nR3 LGB warm-start + T1 scan:')
best_ws_t = None; best_ws_rec = -1
print(f'  {"T":>5}  {"QWK":>7}  {"RecAlta":>9}  Cand')
for T in np.round(np.arange(0.05, 0.51, 0.01), 2):
    r = evaluar_t1('', y_val_v6, proba_lgb_ws, T=T)
    cand_str = 'OK' if r['candidato'] else ''
    if r['candidato'] or (abs(T*100 % 5) < 0.5):
        print(f'  {T:>5.2f}  {r["qwk"]:>7.4f}  {r["rec_alta"]:>9.4f}  {cand_str}')
    if r['candidato'] and r['rec_alta'] > best_ws_rec:
        best_ws_rec = r['rec_alta']; best_ws_t = T; best_ws_r = r

if best_ws_t is not None:
    print(f'\n  R3 mejor candidato T={best_ws_t}: QWK={best_ws_r["qwk"]:.4f}  RecAlta={best_ws_r["rec_alta"]:.4f}')
else:
    print('\n  R3: sin candidato con T1 scan')

print()
print('Comparación directa (transfer learning effect):')
print(f'  LGB v7 base     → QWK={res_lgb_v7_v6["qwk"]:.4f}  RecAlta={res_lgb_v7_v6["rec_alta"]:.4f}')
print(f'  LGB v7 warmstart → QWK={res_lgb_ws["qwk"]:.4f}  RecAlta={res_lgb_ws["rec_alta"]:.4f}')
delta_qwk = res_lgb_ws['qwk'] - res_lgb_v7_v6['qwk']
delta_rec = res_lgb_ws['rec_alta'] - res_lgb_v7_v6['rec_alta']
print(f'  ΔQWK={delta_qwk:+.4f}  ΔRecAlta={delta_rec:+.4f}')

print('\nCelda 7 OK')

R3: LGB v7 Warm-Start sobre v6 train (init_model = lgb_v7_model.txt)
  WS_N_TREES=100  |  WS_LR=0.005  |  N_SUBSAMPLE=1,500,000
  Nota: evaluación LGB_warmstart SOLO (rf_v51_model.pkl no disponible, LOG-50)

Cargando v6 train...
  X_train_ws: (1606527, 100)  NaN: 0
  Normal  : 1,322,633 (82.3%)
  Baja    :   145,311 (9.0%)
  Media   :    32,056 (2.0%)
  Alta    :   106,527 (6.6%)
  Peso Alta/Normal: 12.4x

Entrenando LGB warm-start (100 árboles adicionales, LR=0.005)...
Training until validation scores don't improve for 30 rounds
[2020]	valid_0's multi_logloss: 0.237587
Early stopping, best iteration is:
[2001]	valid_0's multi_logloss: 0.220404
  Entrenado en 5.2 min  |  best_iter extra: 2001
  Guardado: lgb_v7ws_v6_val_proba.npy

  --------------------------------------------------------------------------------
  R3 LGB warm-start (v6 val) — argmax
  --------------------------------------------------------------------------------
  QWK      = 0.5827  ✗   (floor>=0.65)
  RecAlta  = 0.4

In [10]:

print('=' * 92)
print('  TABLA COMPARATIVA v15 — R2 BBSE (LGB v7) + R3 Warm-Start (LGB v7) — v6 val')
print('  [LOG-50: rf_v51_model.pkl no disponible → modelos base = LGB v7]')
print('=' * 92)
print(f'  {"Modelo":<55}  {"QWK":>7}  {"RecAlta":>9}  Cand')
print(f'  {"-"*88}')

rows = [
    ('REFERENCIA: v9 Ens D38 argmax (v5 val)',          res_v9_argmax),
    ('REFERENCIA: v9 Ens D38 + T1=0.38 (v5 val) CAND', res_v9_t138),
    ('─── v6 val ─────────────────────────────────────', None),
    ('LGB v7 base argmax (v6 val, sin calibración)',     res_lgb_v7_v6),
    ('LGB v7 base + T1=0.38 (v6 val)',                  res_lgb_v7_v6_t138),
    ('R2  BBSE argmax: LGB v7 corregido (v6 val)',       res_bbse),
    ('R2b Prior reweighting: LGB v7 (v6 val)',           res_prior),
    ('R3  LGB v7 warm-start argmax (v6 val)',            res_lgb_ws),
]

for nombre, res in rows:
    if res is None:
        print(f'  {nombre}')
        continue
    qwk = res['qwk']; rec = res['rec_alta']; cand = res['candidato']
    qm = '✅' if qwk >= QWK_FLOOR else '✗ '
    rm = '✅' if rec >= RECALTA_FLOOR else '✗ '
    cm = '✅' if cand else '✗ '
    print(f'  {nombre:<55}  {qwk:>7.4f} {qm}  {rec:>7.4f} {rm}  {cm}')

if best_bbse_t is not None:
    print(f'  {"R2  BBSE + T1="+str(best_bbse_t)+" (v6 val)":<55}  {best_bbse_r["qwk"]:>7.4f} ✅  {best_bbse_r["rec_alta"]:>7.4f} ✅  ✅')
if best_ws_t is not None:
    print(f'  {"R3  LGB warmstart + T1="+str(best_ws_t)+" (v6 val)":<55}  {best_ws_r["qwk"]:>7.4f} ✅  {best_ws_r["rec_alta"]:>7.4f} ✅  ✅')

print(f'  {"-"*88}')
print(f'  Floor pre-registrado: QWK>={QWK_FLOOR} AND RecAlta>={RECALTA_FLOOR}  (Kerr 1998)')
print('=' * 92)

print()
candidatos_v15 = [(n, r) for n, r in rows if r is not None and r['candidato']]
if best_bbse_t is not None: candidatos_v15.append((f'R2 BBSE+T1={best_bbse_t}', best_bbse_r))
if best_ws_t is not None:   candidatos_v15.append((f'R3 WS+T1={best_ws_t}', best_ws_r))

if candidatos_v15:
    print(f'  CANDIDATOS v15: {len(candidatos_v15)}')
    for n, r in candidatos_v15:
        print(f'    {n}: QWK={r["qwk"]:.4f}  RecAlta={r["rec_alta"]:.4f}')
    best_v15 = max(candidatos_v15, key=lambda x: x[1]['qwk'])
    print(f'\n  → CANDIDATO FINAL v15: {best_v15[0]}')
    print(f'    QWK={best_v15[1]["qwk"]:.4f}  RecAlta={best_v15[1]["rec_alta"]:.4f}')
    print(f'    Desplaza al candidato provisional (v9 Ens D38 + T1=0.38) para one-shot.')
else:
    print('  v15 SIN CANDIDATO: R2 y R3 no superan floor en v6 val.')
    print('  → Candidato definitivo confirmado: v9 Ens D38 + T1=0.38 (v5 val, LOG-49).')
    print('  → Proceder con: 06 evaluacion test set v7.ipynb')
print()
print('Celda 8 OK')

  TABLA COMPARATIVA v15 — R2 BBSE (LGB v7) + R3 Warm-Start (LGB v7) — v6 val
  [LOG-50: rf_v51_model.pkl no disponible → modelos base = LGB v7]
  Modelo                                                       QWK    RecAlta  Cand
  ----------------------------------------------------------------------------------------
  REFERENCIA: v9 Ens D38 argmax (v5 val)                    0.6529 ✅   0.6509 ✅  ✅
  REFERENCIA: v9 Ens D38 + T1=0.38 (v5 val) CAND            0.6497 ✗    0.6378 ✅  ✗ 
  ─── v6 val ─────────────────────────────────────
  LGB v7 base argmax (v6 val, sin calibración)              0.5946 ✗    0.4583 ✗   ✗ 
  LGB v7 base + T1=0.38 (v6 val)                            0.5864 ✗    0.5012 ✗   ✗ 
  R2  BBSE argmax: LGB v7 corregido (v6 val)                0.0000 ✗    0.0000 ✗   ✗ 
  R2b Prior reweighting: LGB v7 (v6 val)                    0.5880 ✗    0.3594 ✗   ✗ 
  R3  LGB v7 warm-start argmax (v6 val)                     0.5827 ✗    0.4596 ✗   ✗ 
  ------------------------------

In [11]:

import mlflow

try:
    with mlflow.start_run(run_name='r2_bbse_lgb_v7_v6'):
        mlflow.log_params({
            'method': 'BBSE', 'base_model': 'lgb_v7',
            'bbse_reg': BBSE_REG, 'source': 'v5_val', 'target': 'v6_val',
            'note': 'rf_v51_model.pkl unavailable - LOG50'
        })
        mlflow.log_metrics({'qwk_val': res_bbse['qwk'],
                            'recall_alta_val': res_bbse['rec_alta'],
                            'macro_f1_val': res_bbse['macro_f1']})
        if best_bbse_t is not None:
            mlflow.log_params({'t1_best': best_bbse_t})
            mlflow.log_metrics({'qwk_val_t1': best_bbse_r['qwk'],
                                'recall_alta_val_t1': best_bbse_r['rec_alta']})
    print('MLflow r2_bbse_lgb_v7_v6 OK')

    with mlflow.start_run(run_name='r2b_prior_lgb_v7_v6'):
        mlflow.log_params({'method': 'prior_reweighting', 'base_model': 'lgb_v7',
                           'source': 'v5_val', 'target': 'v6_val'})
        mlflow.log_metrics({'qwk_val': res_prior['qwk'],
                            'recall_alta_val': res_prior['rec_alta'],
                            'macro_f1_val': res_prior['macro_f1']})
    print('MLflow r2b_prior_lgb_v7_v6 OK')

    with mlflow.start_run(run_name='r3_lgb_v7_warmstart_v6'):
        mlflow.log_params({
            'method': 'lgb_warmstart', 'init_model': 'lgb_v7',
            'n_trees_extra': WS_N_TREES, 'lr': WS_LR,
            'n_subsample': N_SUBSAMPLE_WS,
            'eval': 'solo_no_ensemble', 'note': 'rf_v51_model.pkl unavailable - LOG50'
        })
        mlflow.log_metrics({'qwk_val': res_lgb_ws['qwk'],
                            'recall_alta_val': res_lgb_ws['rec_alta'],
                            'macro_f1_val': res_lgb_ws['macro_f1']})
        if best_ws_t is not None:
            mlflow.log_params({'t1_best': best_ws_t})
            mlflow.log_metrics({'qwk_val_t1': best_ws_r['qwk'],
                                'recall_alta_val_t1': best_ws_r['rec_alta']})
    print('MLflow r3_lgb_v7_warmstart_v6 OK')

    print('\nv15 completo — revisar tabla comparativa (Celda 8).')
except Exception as e:
    print(f'MLflow error (no crítico): {e}')
    print('v15 completo — revisar tabla comparativa (Celda 8).')

🏃 View run r2_bbse_lgb_v7_v6 at: http://localhost:6000/#/experiments/681062908553326840/runs/cdc8e07e910a4d93a940007b1e3e8392
🧪 View experiment at: http://localhost:6000/#/experiments/681062908553326840
MLflow r2_bbse_lgb_v7_v6 OK
🏃 View run r2b_prior_lgb_v7_v6 at: http://localhost:6000/#/experiments/681062908553326840/runs/0eb401a1e6724e7fa1d007c2c072700b
🧪 View experiment at: http://localhost:6000/#/experiments/681062908553326840
MLflow r2b_prior_lgb_v7_v6 OK
🏃 View run r3_lgb_v7_warmstart_v6 at: http://localhost:6000/#/experiments/681062908553326840/runs/5accc30657e74e9dadfe09484d46b6d0
🧪 View experiment at: http://localhost:6000/#/experiments/681062908553326840
MLflow r3_lgb_v7_warmstart_v6 OK

v15 completo — revisar tabla comparativa (Celda 8).
